<a href="https://colab.research.google.com/github/coretail/Skripsi_Abdullah_Dzaki/blob/main/lampiran_proposal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
file_path = "/content/drive/MyDrive/skripsi abdullah dzaki/data n questionare/Data for SPSS.xlsx"
df = pd.read_excel(file_path)

display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Faculty,Gender,Academic year,Age group,Learning time before COVID-19,Learning time during COVID-19,No. of Siblings,Location of residence,Source of tuition fees,Cumulative average (GPA),...,TM2,TM3,HS1,HS2,HS3,HS4,SE1,SE2,SE3,SE4
0,5,1,2,1,2,2,1,3,1,1,...,3,4,2,4,4,3,4,4,3,4
1,5,1,1,1,1,1,4,3,2,1,...,5,5,5,5,5,2,2,2,2,4
2,5,1,2,1,2,1,2,3,1,1,...,4,4,2,4,4,3,3,4,4,4
3,8,1,3,2,2,2,3,3,1,1,...,4,4,4,4,4,4,4,4,4,4
4,6,1,2,2,2,1,3,2,1,2,...,2,3,5,5,5,5,4,4,1,2


In [ ]:
import numpy as np
from collections import Counter
from math import log
import sys

In [ ]:
# ---------------------------
# Utility: Stratified KFold
# ---------------------------
def stratified_kfold_indices(y, n_splits=5, random_state=42):
    """
    Return list of (train_idx, val_idx) for stratified KFold.
    Simple implementation: split indices per class, then interleave.
    """
    rng = np.random.RandomState(random_state)
    y = np.asarray(y)
    classes = np.unique(y)
    folds = [[] for _ in range(n_splits)]
    for c in classes:
        idxs = np.where(y == c)[0].tolist()
        rng.shuffle(idxs)
        for i, idx in enumerate(idxs):
            folds[i % n_splits].append(idx)
    splits = []
    for i in range(n_splits):
        val_idx = np.array(folds[i])
        train_idx = np.setdiff1d(np.arange(len(y)), val_idx)
        splits.append((train_idx, val_idx))
    return splits

# Logistic Regression
class SimpleLogisticRegression:
    def __init__(self, lr=0.1, n_iter=1000, C=1.0, verbose=False, random_state=42):
        self.lr = lr
        self.n_iter = n_iter
        self.C = C  # inverse regularization strength (like sklearn)
        self.verbose = verbose
        self.random_state = random_state
        self.w = None
        self.b = None

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)
        self.w = rng.normal(scale=0.01, size=n_features)
        self.b = 0.0
        lr = self.lr
        C = self.C
        for it in range(self.n_iter):
            z = X.dot(self.w) + self.b
            p = self._sigmoid(z)
            # gradient
            error = p - y
            grad_w = (X.T.dot(error)) / n_samples + (1.0 / C) * self.w
            grad_b = error.mean()
            # update
            self.w -= lr * grad_w
            self.b -= lr * grad_b
            if self.verbose and it % (self.n_iter//5 + 1) == 0:
                # log loss
                eps = 1e-15
                loss = - (y * np.log(p+eps) + (1-y) * np.log(1-p+eps)).mean() + 0.5*(1.0/C)*(self.w**2).sum()
                print(f"iter {it}/{self.n_iter} loss={loss:.4f}")

    def predict_proba(self, X):
        z = np.asarray(X).dot(self.w) + self.b
        p = self._sigmoid(z)
        return np.vstack([1-p, p]).T

    def predict(self, X, threshold=0.5):
        p = self.predict_proba(X)[:,1]
        return (p >= threshold).astype(int)

# Decision Tree
# CART
class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature_idx = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None  # predicted class at leaf

    def _gini(self, y):
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        ps = counts / counts.sum()
        return 1 - (ps**2).sum()

    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        best_feat, best_thr, best_gain = None, None, 0.0
        parent_gini = self._gini(y)
        if parent_gini == 0:
            return None, None, 0.0
        for feat in range(n_features):
            vals = X[:,feat]
            # consider split points - midpoints of unique sorted values
            uniq = np.unique(vals)
            if len(uniq) <= 1:
                continue
            threshs = (uniq[:-1] + uniq[1:]) / 2.0
            for thr in threshs:
                left_idx = vals <= thr
                right_idx = ~left_idx
                if left_idx.sum() < 1 or right_idx.sum() < 1:
                    continue
                g_left = self._gini(y[left_idx])
                g_right = self._gini(y[right_idx])
                w_left = left_idx.sum() / n_samples
                w_right = right_idx.sum() / n_samples
                gain = parent_gini - (w_left * g_left + w_right * g_right)
                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat
                    best_thr = thr
        return best_feat, best_thr, best_gain

    def fit(self, X, y, depth=0):
        X = np.asarray(X)
        y = np.asarray(y, dtype=int)
        n_samples = len(y)
        num_pos = np.sum(y==1)
        num_neg = np.sum(y==0)
        self.value = 1 if num_pos >= num_neg else 0
        # stopping
        if depth >= self.max_depth or n_samples < self.min_samples_split or self._gini(y) == 0:
            return
        feat, thr, gain = self._best_split(X, y)
        if feat is None:
            return
        self.feature_idx = feat
        self.threshold = thr
        # create children
        left_mask = X[:,feat] <= thr
        right_mask = ~left_mask
        self.left = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
        self.left.fit(X[left_mask], y[left_mask], depth+1)
        self.right = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
        self.right.fit(X[right_mask], y[right_mask], depth+1)

    def _predict_row(self, x):
        if self.feature_idx is None or self.left is None or self.right is None:
            return self.value
        if x[self.feature_idx] <= self.threshold:
            return self.left._predict_row(x)
        else:
            return self.right._predict_row(x)

    def predict(self, X):
        X = np.asarray(X)
        preds = np.array([self._predict_row(row) for row in X], dtype=int)
        return preds

# Random Forest (bootstrap + majority vote)
class SimpleRandomForest:
    def __init__(self, n_estimators=10, max_depth=5, min_samples_split=2, max_features='sqrt', random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.features_idx = []  # bagging

    def _max_features_count(self, n_features):
        if self.max_features == 'sqrt':
            return max(1, int(np.sqrt(n_features)))
        elif self.max_features == 'log2':
            return max(1, int(np.log2(n_features)))
        elif isinstance(self.max_features, float):
            return max(1, int(self.max_features * n_features))
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return n_features

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=int)
        n_samples, n_features = X.shape
        rng = np.random.RandomState(self.random_state)
        m = self._max_features_count(n_features)
        self.trees = []
        self.features_idx = []
        for i in range(self.n_estimators):
            # bootstrap sample
            idxs = rng.randint(0, n_samples, size=n_samples)
            Xb = X[idxs]
            yb = y[idxs]
            # feature bagging
            feat_idx = rng.choice(np.arange(n_features), size=m, replace=False)
            self.features_idx.append(feat_idx)
            tree = DecisionTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit(Xb[:, feat_idx], yb)
            self.trees.append(tree)

    def predict_proba(self, X):
        X = np.asarray(X)
        n_samples = X.shape[0]
        votes = np.zeros((n_samples, self.n_estimators), dtype=int)
        for i, tree in enumerate(self.trees):
            feat_idx = self.features_idx[i]
            preds = tree.predict(X[:, feat_idx])
            votes[:, i] = preds
        # probability = fraction of trees voting 1
        proba = votes.mean(axis=1)
        return np.vstack([1-proba, proba]).T

    def predict(self, X):
        proba = self.predict_proba(X)[:,1]
        return (proba >= 0.5).astype(int)

    # feature importance
    def feature_importances_(self, feature_names):
        cnt = Counter()
        for feat_idx in self.features_idx:
            for f in feat_idx:
                cnt[f] += 1
        tot = sum(cnt.values())
        if tot == 0:
            return {name: 0.0 for name in feature_names}
        imp = {feature_names[i]: cnt[i]/tot for i in range(len(feature_names))}
        return imp

# Confusion metrics
def confusion_matrix(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = np.sum((y_true==1) & (y_pred==1))
    tn = np.sum((y_true==0) & (y_pred==0))
    fp = np.sum((y_true==0) & (y_pred==1))
    fn = np.sum((y_true==1) & (y_pred==0))
    return np.array([[tn, fp],[fn, tp]])

def classification_report(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    acc = (tp+tn)/(tp+tn+fp+fn)
    precision1 = tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall1 = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f11 = 2*precision1*recall1/(precision1+recall1) if (precision1+recall1)>0 else 0.0
    precision0 = tn/(tn+fn) if (tn+fn)>0 else 0.0
    recall0 = tn/(tn+fp) if (tn+fp)>0 else 0.0
    f10 = 2*precision0*recall0/(precision0+recall0) if (precision0+recall0)>0 else 0.0
    return {
        'accuracy': acc,
        'precision_1': precision1, 'recall_1': recall1, 'f1_1': f11,
        'precision_0': precision0, 'recall_0': recall0, 'f1_0': f10
    }

# ROC-AUC
def roc_auc_score(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    # sort by score desc
    desc = np.argsort(-y_score)
    y_true = y_true[desc]
    y_score = y_score[desc]
    # compute TPR/FPR at each threshold
    P = np.sum(y_true==1)
    N = np.sum(y_true==0)
    if P==0 or N==0:
        return 0.5
    tprs = []
    fprs = []
    tp = 0
    fp = 0
    prev_score = None
    for i in range(len(y_true)):
        if y_true[i] == 1:
            tp += 1
        else:
            fp += 1
        tprs.append(tp / P)
        fprs.append(fp / N)
    # add (0,0) at start
    tprs = [0.0] + tprs + [1.0]
    fprs = [0.0] + fprs + [1.0]
    # trapezoid area
    auc = 0.0
    for i in range(1, len(tprs)):
        auc += 0.5 * (tprs[i] + tprs[i-1]) * (fprs[i] - fprs[i-1])
    return auc

# Stacking model
def stacking_train_predict(X, y, base_models_builders, meta_builder, n_folds=5, random_state=42):
    """
    X: numpy array (n_samples, n_features)
    y: numpy int labels
    base_models_builders: list of callables -> model object with fit/predict_proba/predict
      e.g. [lambda: SimpleRandomForest(...), lambda: SimpleLogisticRegression(...)]
    meta_builder: callable -> model object with fit/predict_proba/predict (e.g., SimpleLogisticRegression)
    Returns:
      fitted_base_models (trained on full train data),
      fitted_meta (trained on meta features),
      oof_pred_proba (out-of-fold meta features for train),
      meta_test_proba (meta features on test if provided)
    """
    X = np.asarray(X)
    y = np.asarray(y, dtype=int)
    n_samples = X.shape[0]

    # create OOF meta-features
    n_base = len(base_models_builders)
    oof_meta = np.zeros((n_samples, n_base))
    # generate folds
    folds = stratified_kfold_indices(y, n_splits=n_folds, random_state=random_state)
    # for each fold, train base on train_idx and predict_proba on val_idx
    for i, build in enumerate(base_models_builders):
        for fold_idx, (train_idx, val_idx) in enumerate(folds):
            model = build()
            model.fit(X[train_idx], y[train_idx])
            proba = model.predict_proba(X[val_idx])[:,1]
            oof_meta[val_idx, i] = proba
        # after oof loops, train a final base model on whole data and store
    fitted_base = []
    for build in base_models_builders:
        m = build()
        m.fit(X, y)
        fitted_base.append(m)
    # fit meta on oof_meta
    meta = meta_builder()
    meta.fit(oof_meta, y)
    return fitted_base, meta, oof_meta

# ---------------------------
# Example without demographic data
# ---------------------------
if __name__ == "__main__":
    target_col = "Cumulative average (GPA)"
    likert_cols_manual = ['GS1','GS2','GS3','GS4','GS5','ES1','ES2','ES3','ES4',
                          'TS1','TS2','TS3','TS4','TM1','TM2','TM3','HS1','HS2','HS3','HS4',
                          'SE1','SE2','SE3','SE4']
    # load

    # build X,y (only likert)
    X = df[likert_cols_manual].fillna(df[likert_cols_manual].median()).values.astype(float)
    y = (df[target_col] == 1).astype(int).values

    # standardize (z-score)
    mu = X.mean(axis=0, keepdims=True)
    sigma = X.std(axis=0, keepdims=True)
    sigma[sigma==0] = 1.0
    Xs = (X - mu) / sigma

    # split train/test stratified
    from math import floor
    rng = np.random.RandomState(42)
    idx0 = np.where(y==0)[0]; idx1 = np.where(y==1)[0]
    rng.shuffle(idx0); rng.shuffle(idx1)
    test_frac = 0.2
    n0_test = max(1, int(len(idx0)*test_frac))
    n1_test = max(1, int(len(idx1)*test_frac))
    test_idx = np.concatenate([idx0[:n0_test], idx1[:n1_test]])
    train_idx = np.setdiff1d(np.arange(len(y)), test_idx)
    X_train, y_train = Xs[train_idx], y[train_idx]
    X_test, y_test = Xs[test_idx], y[test_idx]

    # SMOTE
    cnt = Counter(y_train)
    maxc = max(cnt.values())
    ids = []
    for cls, c in cnt.items():
        cls_idx = np.where(y_train==cls)[0]
        if c < maxc:
            extra = rng.choice(cls_idx, size=(maxc-c), replace=True)
            ids.extend(cls_idx.tolist())
            ids.extend(extra.tolist())
        else:
            ids.extend(cls_idx.tolist())
    ids = np.array(ids)
    rng.shuffle(ids)
    X_train_res = X_train[ids]
    y_train_res = y_train[ids]

    # Build base model builders
    def build_rf():
        return SimpleRandomForest(n_estimators=25, max_depth=6, random_state=42)
    def build_lr():
        return SimpleLogisticRegression(lr=0.5, n_iter=800, C=1.0, verbose=False, random_state=42)

    # Train base models & meta via stacking (out-of-fold)
    base_builders = [build_rf, build_lr]
    # create oof meta features using X_train_res and y_train_res
    n_folds = 5
    # Ensure stratified folds on the resampled train
    fits_base = []
    oof_meta = np.zeros((X_train_res.shape[0], len(base_builders)))
    folds = stratified_kfold_indices(y_train_res, n_splits=n_folds, random_state=42)
    for i, build in enumerate(base_builders):
        for train_idx_fold, val_idx_fold in folds:
            m = build()
            m.fit(X_train_res[train_idx_fold], y_train_res[train_idx_fold])
            oof_meta[val_idx_fold, i] = m.predict_proba(X_train_res[val_idx_fold])[:,1]
        # fit base on full resampled train for final use
        m_full = build()
        m_full.fit(X_train_res, y_train_res)
        fits_base.append(m_full)

    # train meta on oof_meta
    meta = SimpleLogisticRegression(lr=0.5, n_iter=1000, C=1.0, verbose=False, random_state=1)
    meta.fit(oof_meta, y_train_res)

    # prepare test meta features: predict_proba from fitted base on X_test
    meta_test = np.zeros((X_test.shape[0], len(fits_base)))
    for i, m in enumerate(fits_base):
        meta_test[:, i] = m.predict_proba(X_test)[:,1]

    # Evaluate base models and stacking
    print("\n EVALUASI PADA TEST SET")
    for name, model in zip(['RF','LR_base'], fits_base):
        proba = model.predict_proba(X_test)[:,1]
        pred = (proba >= 0.5).astype(int)
        cm = confusion_matrix(y_test, pred)
        rpt = classification_report(y_test, pred)
        auc = roc_auc_score(y_test, proba)
        print(f"\nModel: {name}")
        print("Confusion matrix:\n", cm)
        print("Metrics:", rpt)
        print("AUC:", auc)

    # stacking predictions
    proba_stack = meta.predict_proba(meta_test)[:,1]
    pred_stack = (proba_stack >= 0.5).astype(int)
    print("\nModel: Stacking (meta LR)")
    print("Confusion matrix:\n", confusion_matrix(y_test, pred_stack))
    print("Metrics:", classification_report(y_test, pred_stack))
    print("AUC:", roc_auc_score(y_test, proba_stack))

    # Feature importances from RF (rough count-of-feature-usage)
    feature_names = likert_cols_manual
    imp = fits_base[0].feature_importances_(feature_names)
    imp_df = pd.DataFrame(list(imp.items()), columns=['feature','importance']).sort_values('importance', ascending=False)
    print("\nTop features (RF rough importance):")
    print(imp_df.head(15).to_string(index=False))

    # LR base coefficients (readable)
    coef = fits_base[1].w  # weights from simple logistic
    coef_df = pd.DataFrame({'feature': feature_names, 'coef': coef})
    coef_df['abs_coef'] = coef_df['coef'].abs()
    coef_df = coef_df.sort_values('abs_coef', ascending=False)
    print("\nTop LR base coefficients:")
    print(coef_df.head(15).to_string(index=False))

    # Meta learner coefficients
    print("\nMeta-learner coefficients (on meta-features [prob_rf, prob_lr]):")
    print("w:", meta.w, "b:", meta.b)

# End of script


 EVALUASI PADA TEST SET

Model: RF
Confusion matrix:
 [[15 18]
 [30 52]]
Metrics: {'accuracy': np.float64(0.5826086956521739), 'precision_1': np.float64(0.7428571428571429), 'recall_1': np.float64(0.6341463414634146), 'f1_1': np.float64(0.6842105263157895), 'precision_0': np.float64(0.3333333333333333), 'recall_0': np.float64(0.45454545454545453), 'f1_0': np.float64(0.3846153846153846)}
AUC: 0.5218033998521804

Model: LR_base
Confusion matrix:
 [[16 17]
 [26 56]]
Metrics: {'accuracy': np.float64(0.6260869565217392), 'precision_1': np.float64(0.7671232876712328), 'recall_1': np.float64(0.6829268292682927), 'f1_1': np.float64(0.7225806451612903), 'precision_0': np.float64(0.38095238095238093), 'recall_0': np.float64(0.48484848484848486), 'f1_0': np.float64(0.4266666666666667)}
AUC: 0.6053215077605322

Model: Stacking (meta LR)
Confusion matrix:
 [[15 18]
 [30 52]]
Metrics: {'accuracy': np.float64(0.5826086956521739), 'precision_1': np.float64(0.7428571428571429), 'recall_1': np.float64(